> **2026-06-11 note.** This notebook is preserved as the original Month 4 analysis record. Two of its interpretations were later superseded by the Month 5 displacement experiments and the paper (*The Readout, Not the Denoiser*). The closing summary attributes the small deletion log-density to diffusion contraction (E1); the displacement data show the denoiser does not contract the action, and the paper traces the effect to the per-entry quadratic log-density readout (paper Sec. 6.1). The alt-targets section says `l2` clears the O2 bar at −1.23 nats; the `l2` deletion magnitude is in action-space units, not nats, so it does not cross the same bar as the log-density target (paper Sec. 6.2). The executed cells and their outputs are unchanged. A further note on coverage. The E2 section header says all four tasks, while its executed output covers the two tasks whose m=128 step records are released, because the PickCube and StackCube m=128 records were produced on a since-decommissioned pod. Their results are preserved in metrics_report.ipynb and the paper.
>
> **2026-07-01 note.** Two cell sources were pinned so a re-run against the current `data/` reproduces the preserved outputs. The 1B faithfulness glob now selects the seed 42 run this notebook analyzed, since the Month 6 seed 142 and 242 files would otherwise pool in (n=1199 instead of the preserved 320). The alt-target completeness filter now excludes the Month 6 `_logpi` matched-run files, which would otherwise double-count the base logpi rows. The executed outputs are unchanged.
>
> **2026-07-02 note.** The alt-targets section's remaining glosses also predate the Month 6 matched-population run. Its maxdev 50 times improvement figure and its logpi deletion magnitude near −0.003 are superseded by the paper's Table 4 matched-population values, roughly 110 times and −0.0014. The executed outputs are unchanged.

# Month 4 Report: Paper-Strengthening Experiments

Companion to the Month-3 `metrics_report.ipynb` (which holds the frozen Tables 1/2/3 in the Month 3 main-pass record). This notebook covers only the Month-4 experiments that close reviewer-attackable gaps:

- **E1** 1B faithfulness vs 170M
- **E2** m=128 completeness across all 4 tasks
- **E3** C1 model-randomization, 3 variants
- **E4** baseline sensitivity
- **alt-targets** alternative IG targets

Source data: `data/metrics_*.jsonl` (pulled from the 2×4090 pod). Figures pre-rendered by `make_month4_figs.py` into `out/figures_m4/`; this notebook recomputes the tables from the JSONLs for reproducibility.

**Provenance note:** m=64 baselines shown for comparison are the Month-3 record (50-episode runs from the Month-3 main-pass runs, Tables 1/2). Month-4 regenerated runs are fresh (15-episode sidecar regen, 8-episode m=128) — medians are stable across runs, but cell-count percentages on small samples differ from the 50-episode record. Comparisons are directional, not paired.

In [1]:
import json, glob, statistics as st
from pathlib import Path

DATA = Path('..') / 'data'

def load(pat):
    rows = []
    for f in glob.glob(str(DATA / pat)):
        with open(f) as fh:
            for line in fh:
                try: rows.append(json.loads(line))
                except json.JSONDecodeError: pass
    return rows

def med(xs):
    xs = [x for x in xs if x is not None]
    return st.median(xs) if xs else float('nan')

def pct_le(xs, thr=0.03):
    xs = [x for x in xs if x is not None]
    return 100*sum(1 for x in xs if x <= thr)/len(xs) if xs else float('nan')

print('helpers ready; data dir:', DATA.resolve())

helpers ready; data dir: data


## E1 — 1B faithfulness vs 170M (PickCube)

Headline: AUCs *improve* on the working 1B policy, but Δlogπ stays ~0.0002 — refutes the "weak policy" excuse, confirms Δlogπ is structurally insensitive for diffusion policies.

In [2]:
f = load('metrics_faithfulness_*1b_seed42_*.jsonl')
print(f'1B PickCube faithfulness, n={len(f)}')
for mod in ['vision', 'lang']:
    ins = med([r[f'{mod}_insertion_auc'] for r in f])
    dele = med([r[f'{mod}_deletion_auc'] for r in f])
    dlp = med([r[f'{mod}_dlogp_k5'] for r in f])
    print(f'  {mod:6s}: insertion={ins:.3f}  deletion={dele:.3f}  dlogp@k5={dlp:.5f}')
print('\n170M baseline (Month-3 Table 2): vision ins 0.856 / del 0.317; lang ins 0.504 / del 0.477')

1B PickCube faithfulness, n=320
  vision: insertion=0.926  deletion=0.248  dlogp@k5=-0.00016
  lang  : insertion=0.562  deletion=0.283  dlogp@k5=-0.00009

170M baseline (Month-3 Table 2): vision ins 0.856 / del 0.317; lang ins 0.504 / del 0.477


![E1](../out/figures_m4/fig_e1_1b_vs_170m.png)

## E3 — C1 model-randomization, three variants

None of the C1 variants collapse to the ≤0.2 bar; only C2 (input shuffle) does. IG is input-anchored (Adebayo critique of input-multiplied methods). C1-moving and C2 values are the Month-3 record.

In [3]:
for variant, pat in [('frozen ', 'metrics_sanity_C1_frozen_*.jsonl'),
                     ('cascade', 'metrics_sanity_C1_cascade_*.jsonl')]:
    r = load(pat)
    rv = med([x.get('spearman_vision') for x in r])
    rl = med([x.get('spearman_language') for x in r])
    print(f'C1_{variant}: rho_v={rv:.3f}  rho_l={rl:.3f}  (n={len([x for x in r if x.get("spearman_vision") is not None])})')
print('C1_moving (Month-3): rho_v~0.95  rho_l~0.95')
print('C2 input shuffle (Month-3): rho_v~0.13  rho_l~0.12  <- only this passes <=0.2')

C1_frozen : rho_v=0.929  rho_l=0.862  (n=400)
C1_cascade: rho_v=0.713  rho_l=0.910  (n=400)
C1_moving (Month-3): rho_v~0.95  rho_l~0.95
C2 input shuffle (Month-3): rho_v~0.13  rho_l~0.12  <- only this passes <=0.2


![C1 variants](../out/figures_m4/fig_c1_variants.png)

## E2 — m=128 completeness, all four tasks

Vision % of signal-bearing cases (|ref|≥15) with completeness error ≤3%, at m=128.

In [4]:
import os
for f in sorted(glob.glob(str(DATA / 'metrics_*_m128.jsonl'))):
    b = os.path.basename(f)
    if any(x in b for x in ['faithfulness','sanity','baseline']): continue
    rows = [r for r in (json.loads(l) for l in open(f)) if r.get('event')=='step' and r.get('ref_norm_maniskill',0)>=15]
    if not rows: continue
    t, seed = rows[0]['task'], rows[0].get('seed')
    print(f"{t} s{seed}: vis_med={med([r['vision_err'] for r in rows])*100:.2f}%  vis%<=3={pct_le([r['vision_err'] for r in rows]):.1f}  (n={len(rows)})")

PegInsertionSide-v1 s142: vis_med=1.87%  vis%<=3=77.0  (n=178)
PegInsertionSide-v1 s42: vis_med=2.53%  vis%<=3=64.0  (n=189)


PickSingleYCB-v1 s142: vis_med=2.22%  vis%<=3=71.8  (n=195)
PickSingleYCB-v1 s42: vis_med=2.27%  vis%<=3=73.7  (n=194)


![m128 tail](../out/figures_m4/fig_m128_tail.png)

## E4 — baseline sensitivity

Pairwise Spearman ρ of vision patch rankings across black / gray / blur baselines. Solid baselines agree; content-aware blur diverges.

In [5]:
for f in sorted(glob.glob(str(DATA / 'metrics_baseline_sensitivity_*.jsonl'))):
    r = [json.loads(l) for l in open(f)]
    t = r[0]['task']
    print(f"{t}: black-gray={med([x.get('rho_black_gray') for x in r]):.3f}  "
          f"black-blur={med([x.get('rho_black_blur') for x in r]):.3f}  "
          f"gray-blur={med([x.get('rho_gray_blur') for x in r]):.3f}  (n={len(r)})")

PickCube-v1: black-gray=0.825  black-blur=0.539  gray-blur=0.455  (n=30)
StackCube-v1: black-gray=0.814  black-blur=0.546  gray-blur=0.496  (n=30)


![baseline sensitivity](../out/figures_m4/fig_baseline_sens.png)

## Alternative IG targets

Completeness gap AND faithfulness per target on signal-bearing PickCube rows (both seeds, m=64). `logpi` is dead on Δlogp (~−0.003); **`l2` clears the O2 Δlogp bar at −1.23 nats while keeping AUCs faithful** (Ins 0.755, Del 0.284). `maxdev` improves Δ 50× but stays short of the bar. Tradeoff: alt-targets have worse completeness (vis%≤3 ~11% vs `logpi` 77%) from non-smooth sqrt/max aliasing. (l2sq/cosine skipped as diagnostic-only.)

In [6]:
# Alternative IG targets: completeness gap AND faithfulness (the load-bearing result)
import os
def _load(f):
    rows = []
    for l in open(f):
        l = l.strip()
        if not l:
            continue
        try:
            rows.append(json.loads(l))
        except json.JSONDecodeError:
            pass
    return rows
def _faith(tgt):
    vi, vd, vdl = [], [], []
    for f in glob.glob(str(DATA / ('metrics_faithfulness_*_' + tgt + '_*.jsonl'))):
        for r in _load(f):
            vi.append(r.get('vision_insertion_auc'))
            vd.append(r.get('vision_deletion_auc'))
            vdl.append(r.get('vision_dlogp_k5'))
    return med(vi), med(vd), med(vdl), len([x for x in vi if x is not None])
hdr = '{:8s} {:>8s} {:>8s} {:>6s} {:>6s} {:>9s}'.format('target', '|gap|', 'vis%<=3', 'Ins', 'Del', 'dlogp@5')
print(hdr)
for tgt in ['logpi', 'l2', 'maxdev']:
    if tgt == 'logpi':
        files = [f for f in glob.glob(str(DATA / 'metrics_PickCube-v1_170m_seed*.jsonl'))
                 if not any(x in os.path.basename(f) for x in ['faithfulness', 'sanity', 'baseline', 'm128', '_l2', '_maxdev', '_cosine', '_logpi'])]
    else:
        files = glob.glob(str(DATA / ('metrics_PickCube-v1_170m_seed*_' + tgt + '.jsonl')))
    gaps, vpct = [], []
    for f in files:
        rows = [r for r in _load(f) if r.get('event') == 'step' and r.get('ref_norm_maniskill', 0) >= 15]
        gaps += [abs(r.get('vision_gap', 0)) for r in rows]
        vpct += [r['vision_err'] for r in rows]
    fi, fd, fdl, _ = (float('nan'),) * 4 if tgt == 'logpi' else _faith(tgt)
    g = med(gaps)
    vp = (100 * sum(1 for x in vpct if x <= 0.03) / len(vpct)) if vpct else float('nan')
    print('{:8s} {:8.4f} {:7.1f}% {:6.3f} {:6.3f} {:9.4f}'.format(tgt, g, vp, fi, fd, fdl))
print()
print('l2 clears the O2 dlogp bar at -1.23 (bar -0.5) while keeping AUCs faithful.')
print('Tradeoff: alt-targets have worse completeness (vis%<=3 ~11% vs logpi 77%) from non-smooth sqrt/max aliasing.')


target      |gap|  vis%<=3    Ins    Del   dlogp@5
logpi      0.0120    76.8%    nan    nan       nan


l2         3.5194    10.7%  0.755  0.284   -1.2299


maxdev     0.4375    10.9%  0.723  0.273   -0.1562

l2 clears the O2 dlogp bar at -1.23 (bar -0.5) while keeping AUCs faithful.
Tradeoff: alt-targets have worse completeness (vis%<=3 ~11% vs logpi 77%) from non-smooth sqrt/max aliasing.


---
**Summary.** Three Month-3 "misses" are now explained mechanisms: (1) Δlogπ insensitivity = diffusion contraction (E1), (2) C1 non-collapse = IG input-anchoring (E3), (3) baseline divergence = content-aware blur shares input structure (E4). m=128 (E2) confirms the O1 tail is integration-budget-bound on the smooth tasks. Alternative targets recover a live attribution signal where Δlogπ is dead. 